# Accent：banding and coef grouping


In [7]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd

ROOT_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / 'raw-data' / 'accent').is_dir())

sys.path.insert(0, str(ROOT_DIR / 'utilities'))
from transform_audio import freq_band_groupings

In [8]:
TRANSFORM = 'cwt'  # 'fft', 'stft', 'cwt', 'erb'
GROUP = 'gender'  # 'gender', 'onset_group', 'language_group'
KS_THRESHOLD = 0.05
BATCH_SIZE = 1
SUBSAMPLE_EVERY = 100

ACCENT_DIR = ROOT_DIR / 'raw-data' / 'accent'
COEFS_PATH = ACCENT_DIR / 'transforms' / f'{TRANSFORM}_coefs.npz'
DATA_DIR = ROOT_DIR / 'transformed-data-audio' / 'accent'

In [9]:
metadata_df = pd.read_csv(ACCENT_DIR / 'full-accent.csv')
GROUP_NAMES = metadata_df[GROUP].value_counts().index.tolist()
group_files = {
    name: metadata_df.loc[metadata_df[GROUP].eq(name), 'filename'].tolist()
    for name in GROUP_NAMES
}
{name: len(files) for name, files in group_files.items()}


{'male': 844, 'female': 834}

In [10]:
raw_bands = freq_band_groupings(
    COEFS_PATH, ks_threshold=KS_THRESHOLD,
    batch_size=BATCH_SIZE, subsample_every=SUBSAMPLE_EVERY, presplit_depth=1
)
band_sets = {'raw': raw_bands}
{kind: bands for kind, bands in band_sets.items()}

{'raw': [(0, 1),
  (1, 2),
  (2, 3),
  (3, 5),
  (5, 6),
  (6, 7),
  (7, 9),
  (9, 10),
  (10, 11),
  (11, 13),
  (13, 14),
  (14, 15)]}

In [11]:
def load_group_by_band(coefs_path, filenames, transform, bands):
    labels = [f'{start}-{end - 1}' if end > start + 1 else str(start)
              for start, end in bands]
    collected = {label: [] for label in labels}
    with np.load(coefs_path, allow_pickle=True) as archive:
        for filename in filenames:
            coefs = archive[f'{filename}_{transform}']
            for label, (start, end) in zip(labels, bands):
                selected = coefs[start:end]
                if coefs.dtype == object:
                    values = np.concatenate([np.asarray(row, dtype=complex).ravel()
                                             for row in selected])
                else:
                    values = selected.ravel()
                collected[label].append(values)

    grouped = {}
    for label, parts in collected.items():
        values = np.concatenate(parts)
        grouped[label] = {'real': values.real, 'imag': values.imag}
    return grouped

In [12]:
output_dir = DATA_DIR / 'raw-transformed-data'
output_dir.mkdir(parents=True, exist_ok=True)
for group_name in GROUP_NAMES:
    grouped = load_group_by_band(COEFS_PATH, group_files[group_name], TRANSFORM, raw_bands)
    filename = f'accent_Original-Pitch_{TRANSFORM}_{GROUP}_{group_name}.pickle'
    pd.to_pickle({'Original-Pitch': {group_name: grouped}}, output_dir / filename)
    print(filename)
    del grouped


OSError: [Errno 28] No space left on device